# GROUP BY y Agregaciones

In [1]:
import psycopg2

# what is the current total by user?

conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='movies', user='postgres', password='postgres'
)
cursor = conn.cursor()

cursor.execute(
    """
    select 
        c.customer_id, CONCAT(c.first_name, ' ' ,c.last_name) as full_name,
        COUNT(p.payment_id) as number_of_payments,
        SUM(p.amount) as total
    from 
        customer c
    left join 
        payment p on c.customer_id = p.customer_id
    group by
        c.customer_id
    having
        COUNT(p.payment_id) > 35
    Order by 
        total DESC
    ;
    """
)

data = cursor.fetchall()

print(f"{'ID':<10} | {'full name':<25} | {'number of payments':>25} | {'$total':>10}")
print("-" * 100)
for i in data:
    print(f"{i[0]:<10} | {i[1]:<25} | {i[2]:>25} | {i[3]:>25}")
    

ID         | full name                 |        number of payments |     $total
----------------------------------------------------------------------------------------------------
526        | KARL SEAL                 |                        45 |                    221.55
148        | ELEANOR HUNT              |                        46 |                    216.54
144        | CLARA SHAW                |                        42 |                    195.58
178        | MARION SNYDER             |                        39 |                    194.61
137        | RHONDA KENNEDY            |                        39 |                    194.61
459        | TOMMY COLLAZO             |                        38 |                    186.62
469        | WESLEY BULL               |                        40 |                    177.60
468        | TIM CARY                  |                        39 |                    175.61
236        | MARCIA DEAN               |                   

In [2]:
import psycopg2

# how many address exist for each postal_code?

conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='movies', user='postgres', password='postgres'
)
cursor = conn.cursor()

cursor.execute(
    """
    select 
        a.postal_code, count(a.address_id) as total_address
    from 
        address a
    group by
        a.postal_code
    having
        count(a.address_id) > 1
    order by 
        total_address desc
    """
)

data = cursor.fetchall()

for i in data:
    print(i)

('', 4)
('22474', 2)
('9668', 2)
('52137', 2)


In [3]:
import psycopg2

# get top of user with more number of rental?

conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='movies', user='postgres', password='postgres'
)
cursor = conn.cursor()

cursor.execute(
    """
    select 
        concat(c.first_name, ' ', c.last_name) as full_name,
        count(r.rental_id) as total_rental
    from 
        customer c
    left join
        rental r on c.customer_id = r.customer_id
    group by 
        c.customer_id
    order by
        total_rental desc
    limit 5
    """
)

data = cursor.fetchall()
for i in data:
    print(i)

('ELEANOR HUNT', 46)
('KARL SEAL', 45)
('MARCIA DEAN', 42)
('CLARA SHAW', 42)
('TAMMY SANDERS', 41)


## best practices for: GROUP BY y Agregaciones

### 1. Siempre incluye todas las columnas no agregadas en el GROUP BY
- Toda columna en el SELECT que **no** sea una función de agregación (COUNT, SUM, AVG, etc.) **debe** aparecer en el GROUP BY.
- Excepción: PostgreSQL permite omitir columnas si tienen una dependencia funcional directa, pero es buena práctica incluir todas para evitar comportamientos inesperados.

### 2. Usa HAVING para filtrar después de agrupar, WHERE para filtrar antes
- **WHERE** se aplica **antes** del GROUP BY (filtra filas individuales).
- **HAVING** se aplica **después** del GROUP BY (filtra grupos ya formados).
- Ejemplo: `WHERE amount > 0` antes de agregar; `HAVING SUM(amount) > 100` después.

### 3. Favorece WHERE sobre HAVING cuando sea posible
- Filtrar con WHERE reduce el número de filas antes de la agregación, lo cual mejora el rendimiento.
- Ejemplo: usar `WHERE active = true` en lugar de `HAVING COUNT(*) > 0` para excluir inactivos.

### 4. Usa expresiones de agregación consistentes
- **COUNT(*)**: cuenta todas las filas, incluyendo NULLs.
- **COUNT(columna)**: cuenta solo valores no nulos.
- **COUNT(DISTINCT columna)**: cuenta valores únicos no nulos (úsalo con precaución en datasets grandes).
- **SUM(col) / AVG(col)** ignoran NULLs; considera COALESCE si necesitas tratar NULL como 0.

### 5. Orden de ejecución vs. orden de escritura
```
1. FROM / JOIN     → se cargan los datos
2. WHERE           → se filtran filas
3. GROUP BY        → se forman los grupos
4. HAVING          → se filtran grupos
5. SELECT          → se calculan las agregaciones
6. ORDER BY        → se ordenan los resultados
7. LIMIT / OFFSET  → se paginan
```

### 6. Evita funciones pesadas dentro de GROUP BY
- No pongas funciones como `LOWER(name)` o `DATE_TRUNC('day', date)` directamente en el GROUP BY; preprocesa con CTEs o subconsultas.

### 7. Agregaciones sobre JOINs: cuidado con el efecto multiplicativo
- Un JOIN puede duplicar filas antes de agregar. Si une 2 tablas con relación 1:N, el COUNT y SUM se inflan.
- Solución: agrega primero en una subconsulta o CTE, luego haz el JOIN.

### 8. NULLs en GROUP BY
- Los valores NULL se agrupan juntos como un grupo más.
- Usa `COALESCE(col, 'N/A')` si quieres tratar NULLs como categoría visible.

### 9. Limita los resultados con ORDER BY + LIMIT
- Para top-N por grupo (ej. cliente con más compras por país), usa `ORDER BY ... LIMIT 1` junto con `DISTINCT ON` (PostgreSQL) o window functions (`RANK() OVER (PARTITION BY ...)`).

### 10. Usa aliases de columna para legibilidad
- Nombra tus agregaciones: `SUM(amount) AS total_amount` en lugar de `SUM(amount) total_amount`.
- El ORDER BY puede referenciar estos aliases; el HAVING también (en PostgreSQL).
